# MTS MWS: Быстрое обучение Qwen2.5-0.5B (QLoRA) на Kaggle T4 GPU (~8 минут)
Данный ноутбук обучает компактный экстрактор фактов под строгий формат 8-полей (`case_metrics v2.1`).
При запуске на GPU T4 обучение 140 шагов занимает около 7–8 минут.

In [ ]:
# Шаг 1: Установка библиотек (30 секунд)
!pip install -q 'transformers>=4.45.0' 'peft>=0.13.0' 'accelerate>=0.34.0' 'bitsandbytes>=0.44.0' tqdm

In [ ]:
# Шаг 2: Распаковка данных (Kaggle / Colab / локально)
import os, sys, shutil, zipfile, json
from pathlib import Path

# Поиск архивов с данными
for root, dirs, files in os.walk('.'):
    for f in files:
        if f.endswith('.zip') and not os.path.exists('train_sft.jsonl'):
            try:
                with zipfile.ZipFile(os.path.join(root, f), 'r') as z:
                    z.extractall('.')
                print(f'✅ Распакован архив {f}')
            except Exception:
                pass

if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.zip') and not os.path.exists('train_sft.jsonl'):
                try:
                    with zipfile.ZipFile(os.path.join(root, f), 'r') as z:
                        z.extractall('.')
                    print(f'✅ Распакован архив из input: {f}')
                except Exception:
                    pass
            elif f in ['train_sft.jsonl', 'val.jsonl', 'catalog.json', 'case_metrics.py'] and not os.path.exists(f):
                shutil.copy(os.path.join(root, f), f)

assert os.path.exists('train_sft.jsonl'), 'Ошибка: train_sft.jsonl не найден! Загрузите kaggle_train_pack.zip в рабочую папку.'
print('✅ Все данные успешно обнаружены!')

In [ ]:
# Шаг 3: Загрузка модели Qwen2.5-0.5B и настройка LoRA
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
print(f'🚀 Загрузка модели {MODEL_ID}...')

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=compute_dtype
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Шаг 4: Подготовка обучающей выборки с динамическим паддингом
from torch.utils.data import Dataset

class SFTChatDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_len=1280, max_samples=2500):
        self.examples = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            lines = [json.loads(line) for line in f if line.strip()]
        
        for item in lines[:max_samples]:
            msgs = item['messages']
            prompt_msgs = msgs[:2]
            prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
            full_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
            if not full_text.endswith(tokenizer.eos_token):
                full_text += tokenizer.eos_token
                
            prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
            full_ids = tokenizer.encode(full_text, add_special_tokens=False)
            if len(full_ids) > max_len or len(prompt_ids) >= len(full_ids):
                continue
                
            prompt_len = min(len(prompt_ids), len(full_ids))
            labels = [-100] * prompt_len + full_ids[prompt_len:]
            labels = labels[:len(full_ids)]
            self.examples.append({
                'input_ids': full_ids,
                'attention_mask': [1] * len(full_ids),
                'labels': labels
            })
        print(f'✅ Загружено {len(self.examples)} диалогов для ускоренного обучения.')

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

train_dataset = SFTChatDataset('train_sft.jsonl', tokenizer, max_len=1280, max_samples=2000)

def collate_fn(batch):
    max_len = max(len(x['input_ids']) for x in batch)
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    input_ids, attention_mask, labels = [], [], []
    for x in batch:
        pad_len = max_len - len(x['input_ids'])
        input_ids.append(x['input_ids'] + [pad_id] * pad_len)
        attention_mask.append(x['attention_mask'] + [0] * pad_len)
        labels.append(x['labels'] + [-100] * pad_len)
    return {
        'input_ids': torch.tensor(input_ids, dtype=torch.long),
        'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
        'labels': torch.tensor(labels, dtype=torch.long)
    }

In [ ]:
# Шаг 6: Моментальная проверка метрик обученной модели на val.jsonl (case_metrics v2.1)
import copy, time
from difflib import SequenceMatcher
from case_metrics import evaluate, match_facts, load_catalog

print('🔍 Запуск валидации обученной модели...')
load_catalog('catalog.json')
catalog = json.load(open('catalog.json', encoding='utf-8'))
val_calls = [json.loads(line) for line in open('val.jsonl', encoding='utf-8') if line.strip()]
val_gold = {c['call_id']: c for c in val_calls}

# Инференс модели на валидации (прогон первых 100 диалогов для скорости или всех 383)
model.eval()
val_subset = val_calls[:120]  # Быстрый тест 120 звонков (~2.5 мин)
gold_subset = {c['call_id']: c for c in val_subset}

SYSTEM_PROMPT = (
    'Извлекай подтверждённые факты из звонков техподдержки.\n'
    'Выведи только JSON-массив записей, каждая запись имеет 8 полей в таком порядке:\n'
    '[тип объекта, имя объекта, атрибут, значение, полярность, valid_from, valid_to, индекс реплики].\n'
)

raw_preds = {}
t0 = time.time()
print(f'Генерация предсказаний для {len(val_subset)} валидационных диалогов...')

for i, call in enumerate(val_subset):
    turns_text = '\n'.join([f"{t['speaker']} (turn {t['idx']}): {t['text']}" for t in call.get('turns', [])])
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'Диалог:\n{turns_text}\n\nФакты в JSON-формате:'}
    ]
    prompt_str = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_str, return_tensors='pt').to(model.device)
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=384, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    gen_text = tokenizer.decode(gen[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    # Парсинг 8-польного ответа
    start = gen_text.find('[')
    facts = []
    if start >= 0:
        try:
            rows = json.loads(gen_text[start:gen_text.rfind(']')+1])
            for r in rows:
                if isinstance(r, list) and len(r) == 8:
                    facts.append({
                        'object': {'type': str(r[0]), 'name': str(r[1])},
                        'attribute': str(r[2]),
                        'value': str(r[3]),
                        'polarity': int(r[4]),
                        'valid_from': r[5] or call.get('started_at'),
                        'valid_to': r[6],
                        'source': {'turn_idx': int(r[7]), 'span': None, 'quote': None}
                    })
        except Exception:
            pass
    
    c_pred = copy.deepcopy(call)
    c_pred['facts'] = facts
    raw_preds[call['call_id']] = c_pred
    if (i + 1) % 40 == 0:
        print(f'  Обработано {i+1}/{len(val_subset)} диалогов ({time.time()-t0:.1f}с)...')

# Применяем калибратор инвариантов H1-H5
# Оценка через официальный case_metrics
res = evaluate(gold_subset, raw_preds)
print('=' * 70)
print('📊 РЕЗУЛЬТАТЫ ВАЛИДАЦИИ СВЕЖЕОБУЧЕННОЙ МОДЕЛИ (case_metrics v2.1):')
print('=' * 70)
print(f"  Precision       : {res['fact_precision']:.4f}")
print(f"  Recall          : {res['fact_recall']:.4f}")
print(f"  Temporal Recall : {res['temporal_recall']:.4f}  (Порог допуска >= 0.4206)")
print(f"  State Accuracy  : {res['state_accuracy']:.4f}   (Порог допуска >= 0.5145)")
status = '✅ PASS (ДОПУСК ПРОЙДЕН)' if res['temporal_recall'] >= 0.4206 else '⚠️ ТРЕБУЕТСЯ КАЛИБРОВКА'
print(f'  ИТОГОВЫЙ СТАТУС: {status}')
print('=' * 70)

In [ ]:
# Шаг 5: Быстрое обучение на GPU (140 шагов, ~7 минут)
from transformers import Trainer, TrainingArguments
import gc

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
torch.cuda.empty_cache()

model.config.use_cache = False
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()
model.gradient_checkpointing_enable()

training_args = TrainingArguments(
    output_dir='./checkpoints',
    max_steps=140,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=15,
    logging_steps=20,
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    gradient_checkpointing=True,
    report_to='none',
    save_total_limit=1,
    seed=42,
    optim='paged_adamw_8bit'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn
)

print('🚀 Старт дообучения Qwen2.5-0.5B (140 шагов)...')
trainer.train()

# Сохраняем готовый адаптер
output_dir = './final_adapter'
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print('✅ Модель успешно обучена и сохранена в ./final_adapter!')

In [ ]:
# Шаг 6: Упаковка готовых весов в zip для скачивания
!zip -r final_adapter.zip ./final_adapter/
file_size = os.path.getsize('final_adapter.zip')
print(f'📦 Готов архив с весами: final_adapter.zip ({file_size:,} байт)')
print('👉 Скачайте final_adapter.zip через панель файлов справа!')